# Tutorial 5: Difference-in-Differences

Difference-in-differences (DiD) is a popular design for causal inference. In a 2×2 DiD, we compare the change in outcomes for a treated group to the change for a control group. This tutorial shows how to compute DiD on nonlinear (logit) models using smmargins.

## What you will learn

- How to set up a DiD design with a logit outcome model
- How to compute cell predictions, simple effects, and the DiD contrast
- How to compute profile-specific DiD estimates
- How to interpret the results

## The DiD design

In our example, we have:

- `group`: treatment group (`"A"` = control, `"B"` = treated)
- `preexist_Y`: pre-existing condition indicator (0 = without condition, 1 = with condition)
- `condition_X`: binary outcome (presence of a health condition)
- `age`: age in years
- `female`: gender indicator

The pre-existing condition (`preexist_Y`) serves as our "time" dimension: individuals with the condition represent the "post" period, and those without represent the "pre" period. The interaction between `group` and `preexist_Y` captures the DiD effect.

## Step 1: Set up the data and model

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from smmargins import Margins

rng = np.random.default_rng(42)
N = 6_000
df_did = pd.DataFrame({
    "group":      rng.choice(["A", "B"], N, p=[0.55, 0.45]),
    "preexist_Y": rng.integers(0, 2, N),
    "age":        rng.normal(55, 15, N).clip(18, 95),
    "female":     rng.integers(0, 2, N),
})
eta = (-3.5 + 0.04*df_did["age"] - 0.3*df_did["female"]
       + 0.5*(df_did["group"]=="B") + 1.1*df_did["preexist_Y"]
       + 0.8*(df_did["group"]=="B")*df_did["preexist_Y"])
df_did["condition_X"] = (rng.uniform(0,1,N) < 1/(1+np.exp(-eta))).astype(int)

fit_did = smf.logit("condition_X ~ C(group) + preexist_Y + C(group):preexist_Y + age + female",
                    data=df_did).fit(disp=False)
M_did = Margins(fit_did)

The model includes an interaction between group and pre-existing condition, which captures the DiD effect. The coefficient on `C(group)[T.B]:preexist_Y` is the logit-scale interaction.

## Step 2: Compute the DiD

Call the `did()` method with the group variable, the condition ("time") variable, and their levels:

In [ ]:
did = M_did.did("group", "preexist_Y",
                group_levels=["A", "B"],
                condition_levels=[0, 1])

The `did` object contains three components:

1. `did.cells`: Predictions in each of the four cells
2. `did.simple_effects`: First differences (simple effects)
3. `did.did`: The difference-in-differences contrast

## Step 3: View the four cells

In [ ]:
did.cells.summary()

These are the predicted probabilities of `condition_X` in each of the four group-condition combinations. Notice that the outcome is higher for those with `preexist_Y = 1` in both groups, and higher in group B overall.

## Step 4: View the simple effects

The simple effects show the change from `preexist_Y = 0` to `preexist_Y = 1` within each group:

In [ ]:
did.simple_effects.summary()

In group A (control), the predicted probability increases by 25.5 percentage points. In group B (treated), it increases by 33.9 percentage points.

## Step 5: View the DiD contrast

The DiD contrast subtracts the simple effect in group A from the simple effect in group B:

In [ ]:
did.did.summary()

The DiD estimate is 0.0835. This means that the increase in the predicted probability of `condition_X` associated with having `preexist_Y = 1` is 8.35 percentage points larger in group B than in group A. This is the causal effect of interest under the parallel trends assumption.

## Step 6: Profile-specific DiD

The DiD estimate can vary by covariate profile. We can compute the DiD at specific values of age and gender using `atexog`:

In [ ]:
did_profile = M_did.did("group", "preexist_Y",
                        group_levels=["A", "B"],
                        condition_levels=[0, 1],
                        atexog={"age": 60, "female": 0})

In [ ]:
did_profile.cells.summary()

In [ ]:
did_profile.did.summary()

For a 60-year-old male, the DiD estimate is 0.0938, slightly larger than the population-averaged estimate of 0.0835. This illustrates how the treatment effect can vary across covariate profiles.

## Summary: the DiD workflow

1. Fit a model with the group-condition interaction
2. Call `M.did()` with group and condition variables
3. Inspect `cells`, `simple_effects`, and `did` tables
4. Use `atexog` for profile-specific estimates

## Recap

In this tutorial we:

1. Set up a 2×2 DiD design with a logit outcome model
2. Computed predictions in the four cells
3. Computed simple effects (first differences) within each group
4. Computed the DiD contrast (difference of differences)
5. Showed how to get profile-specific DiD estimates with `atexog`

## Next steps

- Learn about counterfactual predictions and plotting in {doc}`Tutorial 6: Counterfactuals and Plotting </tutorials/counterfactuals_and_plotting>`
- Read the reference for the {doc}`did() method </api>` and {doc}`DiD result object </api>`
- Understand the Ai-Norton approach to DiD on nonlinear models in {doc}`Explanation: Ai-Norton DiD </explanations/ai_norton_did>`
- Learn about joint tests and pairwise comparisons in {doc}`How-To: Joint Tests and Pairwise Comparisons </howto/joint_tests_pairwise>`